In [15]:
import json
import os 

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "martin2013memory")
original_data_pathway = os.path.join(pathway, "original_data")


In [16]:

# complete_path_1 = os.path.join(original_data_pathway, "martinordas2013memory_general event_reminder.csv")
# complete_path_2 = os.path.join(original_data_pathway, "martinordas2013memory_unique event_reminder.csv")


complete_path_1 = os.path.join(original_data_pathway, "reminder_general.csv")
complete_path_2 = os.path.join(original_data_pathway, "reminder_unique.csv")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [17]:
import pandas as pd
import numpy as np
import pyreadstat

df1 = pd.read_csv(complete_path_1)
df2 = pd.read_csv(complete_path_2)

experiment_import = [[df1, 'general_event', '1'],
                    [df2, 'unique_event', '2']]
for x,y,k in experiment_import:
    x['experiment_name']=y
    x['experiment']=k

In [18]:
df1['latency approach first box'] = df1['latency approach first box'].str.replace('excluded: baby found tools', 'excluded-baby_found_tools')

# df2['experience'].unique()
df2.rename(columns={"experience":'experience_temp'}, inplace=True)

df2_list = [['GE+UE','general_event_before_unique'], 
            ['UE', 'unique_event_only'], 
            ['GE', 'general_event_only']]
for x,y in df2_list:
    df2.loc[df2.experience_temp == x, ['experience']] = y


In [19]:
data_frames=[df1, df2]
for index, x in enumerate(data_frames):
    x.columns = map(str.lower, x.columns)
    x=x.applymap(lambda s: s.lower() if type(s) == str else s) 
    x.rename(columns={"subject": "ape",
        "species":"species_x",
        "find tools":"find_tools",
        "solve task":"solve_task",
        "latency approach first box":"latency_approach_first_box",
        "left room":"left_room",
        "food room":"food_room",
        "midle right":"middle_right",
        "right room":"right_room",
        "group":"group_original"}, inplace=True)
    x['ape'] = x['ape'].str.rstrip()
    x['study_id']="martin2013memory"
    data_frames[index]=x
new_df=data_frames[0]
fulldf = pd.concat(data_frames, ignore_index=True, sort=False)


In [20]:
fulldf[['ape','ape_2']] = fulldf['ape'].str.split('/',expand=True)
# fulldf.columns


In [21]:

comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")
df_name  = pd.read_csv(comp_path_name_errors)
fulldf['ape'] = fulldf['ape'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    fulldf['ape'].replace(x, y, inplace=True)
    fulldf['ape_2'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)    
fulldf= fulldf.merge(apedf,left_on='ape', right_on='name', how='left')

# comp_path_ape_info_2 = os.path.join(pathway_gen, "apes_includeindatabase_2.csv")
# apedf_2 = pd.read_csv(comp_path_ape_info_2)
# fulldf= fulldf.merge(apedf_2,left_on='ape_2', right_on='name_2', how='left')



In [22]:
# role=[]
# role_2=[]
# for index, row in fulldf.iterrows():
#     if not pd.isna(row['ape']):
#         role.append("subject")
#     else:
#         role.append("")
# fulldf = fulldf.assign(role=role)
# for index, row in fulldf.iterrows(): 
#     if not pd.isna(row['ape_2']):
#         role_2.append("ape_2")
#     else:
#         role_2.append("")
# fulldf = fulldf.assign(role_2=role_2)



In [23]:
# dyad=[]
# for index, row in fulldf.iterrows():
#     if not pd.isna(row['ape_2']):
#         dyad.append(row['ape']+"_"+ row['ape_2'])
#     else:
#         dyad.append("")
# fulldf = fulldf.assign(dyad=dyad)

import re
replace_list = ['experience']
replace_1=re.compile('( |\+|\:)') 
replace_2=re.compile('(\ï)') 
for x in replace_list:
        fulldf[x] = fulldf[x].str.replace(replace_1, '_')
        fulldf[x] = fulldf[x].str.replace(replace_2, 'i')
        fulldf[x] = fulldf[x].str.replace('__', '_')
# fulldf['latency_approach_first_box'] = fulldf['latency_approach_first_box'].str.replace('excluded: baby found tools', 'excluded-baby_found_tools')

In [24]:
# fulldf.columns
fulldf.rename(columns={"ape": "participant", 
    "ape_2":"present", 
    "correct location":"correct_location",
    'strategies ':'strategies_temp'}, inplace=True)

fulldf.dropna(subset=['participant'], inplace=True)

In [25]:
strategies_list = [[1.0,'took_and_used_correct_tool'], 
            [3.0, 'took_incorrect_tool_then_immediately_took_and_used_correct_tool'], 
            [4.0, 'passive_drop_out_participant']]
for x,y in strategies_list:
    fulldf.loc[fulldf.strategies_temp == x, ['strategies']] = y

fulldf.loc[fulldf.experience == 'passive', ['drop_out']] = 'true'

In [26]:
complete_path_age = os.path.join(original_data_pathway, "subject_list.csv")
subject_list = pd.read_csv(complete_path_age)   
fulldf= fulldf.merge(subject_list,left_on='participant', right_on='name', how='left')
fulldf.rename(columns={"age": "age_in_years"}, inplace=True)

In [27]:
fulldf=fulldf[['study_id', 'experiment', 'experiment_name',
        'participant' ,'age_in_years','sex', 'present' ,'species',
         'group_original',  
        'experience', 'find_tools', 'solve_task',
       'latency_approach_first_box', 'left_room', 'food_room',
       'middle_right', 'right_room', 'strategies', 
        'correct_location', 'drop_out']]


In [28]:
for index in range(1,3):
    exp = fulldf[fulldf['experiment'] == str(index)]
    exp = exp.dropna(axis=1, how='all')
    comp_out_path = os.path.join(out_pathway, 'martin2013memory_exp'+str(index)+'_standardized.csv')
    exp.to_csv(comp_out_path, encoding='utf-8-sig', index=False)
    names = exp.columns.tolist()
    exp_g = pd.DataFrame(names)
    exp_g = exp_g.rename(columns={0: "column_name"})
    exp_g["description"] = ""
    exp_g=exp_g[["column_name", "description"]]
    comp_out_path_glossary = os.path.join(out_pathway, 'martin2013memory_exp'+str(index)+'_glossary.csv')
    exp_g.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)